In [1]:
import sys
sys.path.append('../input/iterative-stratification/iterative-stratification-master')
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook

- train_features.csv - Features for the training set. Features `g-` signify gene expression data, and `c-` signify cell viability data. `cp_type` indicates samples treated with a compound (`cp_vehicle`) or with a control perturbation (`ctrl_vehicle`); control perturbations have no MoAs; `cp_time` and `cp_dose` indicate treatment duration (24, 48, 72 hours) and `dose` (high or low).
- train_targets_scored.csv - The binary MoA targets that are scored.
- train_targets_nonscored.csv - Additional (optional) binary MoA responses for the training data. These are not predicted nor scored.
- test_features.csv - Features for the test data. You must predict the probability of each scored MoA for each row in the test data.
- sample_submission.csv - A submission file in the correct format.

In [3]:
train=pd.read_csv("/kaggle/input/lish-moa/train_features.csv")
train

,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,trt_cp,48,D1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,trt_cp,72,D2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23809,id_fffb1ceed,trt_cp,24,D2,0.1394,-0.0636,-0.1112,-0.5080,-0.4713,0.7201,...,0.1969,0.0262,-0.8121,0.3434,0.5372,-0.3246,0.0631,0.9171,0.5258,0.4680
23810,id_fffb70c0c,trt_cp,24,D2,-1.3260,0.3478,-0.3743,0.9905,-0.7178,0.6621,...,0.4286,0.4426,0.0423,-0.3195,-0.8086,-0.9798,-0.2084,-0.1224,-0.2715,0.3689
23811,id_fffc1c3f4,ctl_vehicle,48,D2,0.3942,0.3756,0.3109,-0.7389,0.5505,-0.0159,...,0.5409,0.3755,0.7343,0.2807,0.4116,0.6422,0.2256,0.7592,0.6656,0.3808
23812,id_fffcb9e7c,trt_cp,24,D1,0.6660,0.2324,0.4392,0.2044,0.8531,-0.0343,...,-0.1105,0.4258,-0.2012,0.1506,1.5230,0.7101,0.1732,0.7015,-0.6290,0.0740


In [4]:
train_target = pd.read_csv("../input/lish-moa/train_targets_scored.csv")
train_target

,sig_id,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,id_000644bb2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,id_000779bfc,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,id_000a6266a,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,id_0015fd391,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,id_001626bd3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23809,id_fffb1ceed,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23810,id_fffb70c0c,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23811,id_fffc1c3f4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23812,id_fffcb9e7c,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Get indices of Ctl vehicle

In [5]:
ctlVehicle_idx = train["cp_type"] != "ctl_vehicle"
train = train.loc[ctlVehicle_idx].reset_index(drop=True)
train = train.drop("cp_type", axis=1)
train_target = train_target.loc[ctlVehicle_idx].reset_index(drop=True)

In [6]:
gcols = [g for g in train.columns if "g-" in g]
ccols = [c for c in train.columns if "c-" in c]
cpcols = [cp for cp in train.columns if "cp_" in cp]

# Encode categorical features

In [7]:
from sklearn.preprocessing import LabelEncoder

test = pd.read_csv("/kaggle/input/lish-moa/test_features.csv")
ctlVehicle_test = test["cp_type"] == "ctl_vehicle"

test = test.drop("cp_type", axis=1)

train["cp_time"]=train["cp_time"].map({24: 1, 48: 2, 72: 3})
train["cp_dose"] = train["cp_dose"].map({"D1": 1, "D2": 2})
test["cp_time"]= test["cp_time"].map({24: 1, 48: 2, 72: 3})
test["cp_dose"]=test["cp_dose"].map({"D1": 1, "D2": 2})
train

,sig_id,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,g-6,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,1,1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,-1.0220,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,3,1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,0.2341,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,2,1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,0.1715,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,2,1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,-1.9590,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,3,2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,-0.2800,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21943,id_fff8c2444,3,1,0.1608,-1.0500,0.2551,-0.2239,-0.2431,0.4256,-0.1166,...,0.0789,0.3538,0.0558,0.3377,-0.4753,-0.2504,-0.7415,0.8413,-0.4259,0.2434
21944,id_fffb1ceed,1,2,0.1394,-0.0636,-0.1112,-0.5080,-0.4713,0.7201,0.5773,...,0.1969,0.0262,-0.8121,0.3434,0.5372,-0.3246,0.0631,0.9171,0.5258,0.4680
21945,id_fffb70c0c,1,2,-1.3260,0.3478,-0.3743,0.9905,-0.7178,0.6621,-0.2252,...,0.4286,0.4426,0.0423,-0.3195,-0.8086,-0.9798,-0.2084,-0.1224,-0.2715,0.3689
21946,id_fffcb9e7c,1,1,0.6660,0.2324,0.4392,0.2044,0.8531,-0.0343,0.0323,...,-0.1105,0.4258,-0.2012,0.1506,1.5230,0.7101,0.1732,0.7015,-0.6290,0.0740


In [8]:
train

,sig_id,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,g-6,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,1,1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,-1.0220,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,3,1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,0.2341,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,2,1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,0.1715,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,2,1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,-1.9590,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,3,2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,-0.2800,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21943,id_fff8c2444,3,1,0.1608,-1.0500,0.2551,-0.2239,-0.2431,0.4256,-0.1166,...,0.0789,0.3538,0.0558,0.3377,-0.4753,-0.2504,-0.7415,0.8413,-0.4259,0.2434
21944,id_fffb1ceed,1,2,0.1394,-0.0636,-0.1112,-0.5080,-0.4713,0.7201,0.5773,...,0.1969,0.0262,-0.8121,0.3434,0.5372,-0.3246,0.0631,0.9171,0.5258,0.4680
21945,id_fffb70c0c,1,2,-1.3260,0.3478,-0.3743,0.9905,-0.7178,0.6621,-0.2252,...,0.4286,0.4426,0.0423,-0.3195,-0.8086,-0.9798,-0.2084,-0.1224,-0.2715,0.3689
21946,id_fffcb9e7c,1,1,0.6660,0.2324,0.4392,0.2044,0.8531,-0.0343,0.0323,...,-0.1105,0.4258,-0.2012,0.1506,1.5230,0.7101,0.1732,0.7015,-0.6290,0.0740


In [9]:
test

,sig_id,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,g-6,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_0004d9e33,1,1,-0.5458,0.1306,-0.5135,0.4408,1.5500,-0.1644,-0.2140,...,0.0981,0.7978,-0.1430,-0.2067,-0.2303,-0.1193,0.0210,-0.0502,0.1510,-0.7750
1,id_001897cda,3,1,-0.1829,0.2320,1.2080,-0.4522,-0.3652,-0.3319,-1.8820,...,-0.1190,-0.1852,-1.0310,-1.3670,-0.3690,-0.5382,0.0359,-0.4764,-1.3810,-0.7300
2,id_002429b5b,1,1,0.1852,-0.1404,-0.3911,0.1310,-1.4380,0.2455,-0.3390,...,-0.2261,0.3370,-1.3840,0.8604,-1.9530,-1.0140,0.8662,1.0160,0.4924,-0.1942
3,id_00276f245,1,2,0.4828,0.1955,0.3825,0.4244,-0.5855,-1.2020,0.5998,...,0.1260,0.1570,-0.1784,-1.1200,-0.4325,-0.9005,0.8131,-0.1305,0.5645,-0.5809
4,id_0027f1083,2,1,-0.3979,-1.2680,1.9130,0.2057,-0.5864,-0.0166,0.5128,...,0.4965,0.7578,-0.1580,1.0510,0.5742,1.0900,-0.2962,-0.5313,0.9931,1.8380
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3977,id_ff7004b87,1,1,0.4571,-0.5743,3.3930,-0.6202,0.8557,1.6240,0.0640,...,-1.1790,-0.6422,-0.4367,0.0159,-0.6539,-0.4791,-1.2680,-1.1280,-0.4167,-0.6600
3978,id_ff925dd0d,1,1,-0.5885,-0.2548,2.5850,0.3456,0.4401,0.3107,-0.7437,...,0.0210,0.5780,-0.5888,0.8057,0.9312,1.2730,0.2614,-0.2790,-0.0131,-0.0934
3979,id_ffb710450,3,1,-0.3985,-0.1554,0.2677,-0.6813,0.0152,0.4791,-0.0166,...,0.4418,0.9153,-0.1862,0.4049,0.9568,0.4666,0.0461,0.5888,-0.4205,-0.1504
3980,id_ffbb869f2,2,2,-1.0960,-1.7750,-0.3977,1.0160,-1.3350,-0.2207,-0.3611,...,0.3079,-0.4473,-0.8192,0.7785,0.3133,0.1286,-0.2618,0.5074,0.7430,-0.0484


In [10]:
train_target = train_target.drop("sig_id", axis=1)
train_target

,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,adrenergic_receptor_agonist,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21943,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21944,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
train = train.drop("sig_id", axis=1)
test = test.drop("sig_id", axis=1)

In [14]:
import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow_addons as tfa

def create_shallow_model(out_shape=206):
    inp = tf.keras.Input(shape=(train.shape[1],))
    x = tfa.layers.WeightNormalization(L.Dense(train.shape[1]))(inp)
    x = L.BatchNormalization()(x)
    x = L.LeakyReLU(alpha=0.005)(x)
    x = L.Dropout(0.2)(x)
    x = tfa.layers.WeightNormalization(L.Dense(128))(x)
    x = L.BatchNormalization()(x)
    x = L.LeakyReLU(alpha=0.005)(x)
    x = L.Dropout(0.2)(x)
    output = tfa.layers.WeightNormalization(L.Dense(out_shape, activation="sigmoid"))(x)
    model = tf.keras.Model(inputs=inp, outputs=output)
    
    sgd = tf.keras.optimizers.SGD()
    adamw = tfa.optimizers.AdamW(weight_decay = 0.0001)
    adam = tf.keras.optimizers.Adam()
    radam = tfa.optimizers.RectifiedAdam()
    lookahead_radam = tfa.optimizers.Lookahead(radam)
    lookahead_adamw = tfa.optimizers.Lookahead(adamw)
    
    model.compile(loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=1e-6), optimizer=adamw)
    return model

def create_mid_model(out_shape=206):
 
    inp = tf.keras.Input(shape=(train.shape[1],))
    x = tfa.layers.WeightNormalization(L.Dense(train.shape[1]))(inp)
    x = L.BatchNormalization()(x)
    x = L.LeakyReLU(alpha=0.001)(x)
    x = L.Dropout(0.3)(x)
    x = tfa.layers.WeightNormalization(L.Dense(512))(x)
    x = L.BatchNormalization()(x)
    x = L.LeakyReLU(alpha=0.001)(x)
    x = L.Dropout(0.3)(x)
    x = tfa.layers.WeightNormalization(L.Dense(512))(x)
    x = L.BatchNormalization()(x)
    x = L.LeakyReLU(alpha=0.001)(x)
    x = L.Dropout(0.2)(x)
    output = tfa.layers.WeightNormalization(L.Dense(out_shape, activation="sigmoid"))(x)
    model = tf.keras.Model(inputs=inp, outputs=output)
    
    sgd = tf.keras.optimizers.SGD(momentum=0.9)
    adamw = tfa.optimizers.AdamW(weight_decay = 0.001)
    adam = tf.keras.optimizers.Adam()
    radam = tfa.optimizers.RectifiedAdam()
    lookahead_radam = tfa.optimizers.Lookahead(radam)
    lookahead_adamw = tfa.optimizers.Lookahead(adamw)
    
    model.compile(loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=1e-6), optimizer=radam)
    return model

def create_deep_model(out_shape=206):

    inp = tf.keras.Input(shape=(train.shape[1],))
    x = tfa.layers.WeightNormalization(L.Dense(train.shape[1]))(inp)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Dropout(0.3)(x)
    x = tfa.layers.WeightNormalization(L.Dense(512))(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Dropout(0.42)(x)
    x = tfa.layers.WeightNormalization(L.Dense(512))(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Dropout(0.3)(x)
    x = tfa.layers.WeightNormalization(L.Dense(256))(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Dropout(0.3)(x)
    x = tfa.layers.WeightNormalization(L.Dense(256))(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Dropout(0.2)(x)
    output = tfa.layers.WeightNormalization(L.Dense(out_shape, activation="sigmoid"))(x)
    model = tf.keras.Model(inputs=inp, outputs=output)
    
    sgd = tf.keras.optimizers.SGD()
    adamw = tfa.optimizers.AdamW(weight_decay = 0.0001)
    adam = tf.keras.optimizers.Adam()
    radam = tfa.optimizers.RectifiedAdam()
    lookahead_radam = tfa.optimizers.Lookahead(radam)
    lookahead_adamw = tfa.optimizers.Lookahead(adamw)
    
    model.compile(loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=1e-6), optimizer=adamw)
    return model

In [16]:
# from sklearn.model_selection import KFold

predictions = []
kf = MultilabelStratifiedKFold(n_splits=5)
tf.random.set_seed(42)
for fold_id, (train_idx, valid_idx) in enumerate(kf.split(train, train_target)):
    model2 = create_mid_model()
    history2 = model2.fit(train.iloc[train_idx], train_target.iloc[train_idx], batch_size=32,
              validation_data=(train.iloc[valid_idx], train_target.iloc[valid_idx]),
             epochs=200,
             verbose=2,
             callbacks=[
    tf.keras.callbacks.ReduceLROnPlateau(),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("model2_fold" + str(fold_id) + ".h5", save_best_only=True, save_weights_only=True)
])
    print("Model 2, Fold ID: {}, train loss: {}, valid loss: {}".format(fold_id, min(history2.history["loss"]), min(history2.history["val_loss"])))
    model2 = create_mid_model()
    model2.load_weights("model2_fold" + str(fold_id) + ".h5")
    pred2 = model2.predict(test)
    predictions.append(pred2)

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass shuffle=False, random_state=None as keyword args. From version 0.25 passing these as positional arguments will result in an error
  FutureWarning)


Epoch 1/200
549/549 - 6s - loss: 0.1175 - val_loss: 0.0286
Epoch 2/200
549/549 - 6s - loss: 0.0202 - val_loss: 0.0207
Epoch 3/200
549/549 - 6s - loss: 0.0184 - val_loss: 0.0179
Epoch 4/200
549/549 - 6s - loss: 0.0175 - val_loss: 0.0176
Epoch 5/200
549/549 - 6s - loss: 0.0168 - val_loss: 0.0172
Epoch 6/200
549/549 - 6s - loss: 0.0161 - val_loss: 0.0171
Epoch 7/200
549/549 - 7s - loss: 0.0156 - val_loss: 0.0168
Epoch 8/200
549/549 - 6s - loss: 0.0149 - val_loss: 0.0169
Epoch 9/200
549/549 - 6s - loss: 0.0144 - val_loss: 0.0170
Epoch 10/200
549/549 - 6s - loss: 0.0137 - val_loss: 0.0171
Epoch 11/200
549/549 - 6s - loss: 0.0131 - val_loss: 0.0172
Epoch 12/200
549/549 - 6s - loss: 0.0124 - val_loss: 0.0172
Epoch 13/200
549/549 - 6s - loss: 0.0118 - val_loss: 0.0175
Epoch 14/200
549/549 - 6s - loss: 0.0111 - val_loss: 0.0179
Epoch 15/200
549/549 - 6s - loss: 0.0105 - val_loss: 0.0180
Epoch 16/200
549/549 - 6s - loss: 0.0099 - val_loss: 0.0185
Epoch 17/200
549/549 - 6s - loss: 0.0092 - val_lo

In [17]:
pred = np.mean(predictions, axis=0)
# pred = np.clip(pred, 0.001, 0.999)
pred.shape

(3982, 206)

In [22]:
for i in range(len(pred)):
    for j in range(len(pred[0])):
        if pred[i][j] < 0.5:
            pred[i][j] = 0.001
        else:
            pred[i][j] = 0.999
pred

array([[0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001],
       [0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001],
       [0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001],
       ...,
       [0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001],
       [0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001],
       [0.001, 0.001, 0.001, ..., 0.001, 0.001, 0.001]], dtype=float32)

In [23]:
sub = pd.read_csv("../input/lish-moa/sample_submission.csv")
sub.loc[:, 1:] = pred
sub.loc[ctlVehicle_test, sub.columns != "sig_id"] = 0

# sub.loc[:, 1:] = tf.keras.utils.normalize(pred)
sub

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:2: FutureWarning: Slicing a positional slice with .loc is not supported, and will raise TypeError in a future version.  Use .loc with labels or .iloc with positions instead.
  


,sig_id,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
0,id_0004d9e33,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
1,id_001897cda,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
2,id_002429b5b,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,id_00276f245,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
4,id_0027f1083,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3977,id_ff7004b87,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
3978,id_ff925dd0d,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
3979,id_ffb710450,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001
3980,id_ffbb869f2,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,...,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001


In [24]:
sub.to_csv("submission.csv", index=False)